## 🔍 PASO 10 — Explicabilidad con Tree SHAP

In [ ]:
print('Calculando valores SHAP (puede tardar 2–5 minutos)...')
explainer = shap.TreeExplainer(rf)
X_shap    = Xte_s[:300]          # Submuestra para eficiencia
sv_raw    = explainer.shap_values(X_shap)

# Extraer valores para la clase positiva
if isinstance(sv_raw, list):
    sv = sv_raw[1]
elif sv_raw.ndim == 3:
    sv = sv_raw[:, :, 1]
else:
    sv = sv_raw

print(f'✓ Valores SHAP calculados. Forma: {sv.shape}')

# ── Gráfico 1: Importancia global (barras) ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
mean_abs  = np.abs(sv).mean(axis=0)
order     = np.argsort(mean_abs)

ax = axes[0]
ax.barh([FEAT_LABELS[i] for i in order], mean_abs[order], color=C['s'])
ax.set_xlabel('Importancia SHAP media (|valor|)', fontsize=11)
ax.set_title('Explicabilidad global\nRandom Forest — Tree SHAP',
              fontsize=12, fontweight='bold', color=C['p'])

# ── Gráfico 2: Beeswarm ───────────────────────────────────────
plt.sca(axes[1])
shap.summary_plot(sv, X_shap, feature_names=FEAT_LABELS,
                  show=False, plot_size=None)
plt.title('Distribución de valores SHAP', fontsize=12,
           fontweight='bold', color=C['p'])

plt.tight_layout()
plt.savefig('shap_explicabilidad.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Explicación local: un paciente de alto riesgo ─────────────
print('\n=== Explicación local — Paciente de alto riesgo (índice 5) ===')
idx      = 5
sv_local = sv[idx]
prob_loc = rf.predict_proba(X_shap[idx:idx+1])[0][1]
print(f'  Probabilidad de readmisión: {prob_loc*100:.1f} %')
print('  Contribución de cada variable (top 8):')
pares = sorted(zip(FEAT_LABELS, sv_local), key=lambda x: abs(x[1]), reverse=True)[:8]
for lbl, val in pares:
    signo = '▲ aumenta riesgo' if val > 0 else '▼ reduce riesgo'
    print(f'    {lbl:<22}: {val:+.4f}  ({signo})')

## 🖥️ PASO 11 — Sistema de Predicción Interactivo

Ingresa los datos de un paciente y obtén la predicción junto con la explicación SHAP.

In [ ]:
def predecir_paciente(
    time_in_hospital=5,
    n_lab_procedures=45,
    n_procedures=1,
    n_medications=18,
    n_outpatient=0,
    n_inpatient=2,        # ← Variable más importante
    n_emergency=0,
    age_enc=3,            # 0=[40-50) … 5=[90-100)
    glucose_test_enc=0,   # 0=sin test, 1=normal, 2=alto
    A1Ctest_enc=2,        # 0=sin test, 1=normal, 2=alto
    change_enc=1,         # 0=no, 1=sí
    diabetes_med_enc=1,   # 0=no, 1=sí
    medical_specialty_enc=3,
    diag_1_enc=2,
    diag_2_enc=1,
    diag_3_enc=4,
    modelo=rf
):
    """
    Predice la probabilidad de readmisión hospitalaria para un paciente
    diabético y genera la explicación mediante valores SHAP.

    Parámetros
    ----------
    Todos los atributos clínicos del paciente (ver nombres de variables).
    modelo : estimador entrenado (por defecto, Random Forest).

    Retorna
    -------
    dict con 'readmitido', 'probabilidad', 'riesgo' y 'shap_values'.
    """
    valores = [
        time_in_hospital, n_lab_procedures, n_procedures, n_medications,
        n_outpatient, n_inpatient, n_emergency, age_enc,
        glucose_test_enc, A1Ctest_enc, change_enc, diabetes_med_enc,
        medical_specialty_enc, diag_1_enc, diag_2_enc, diag_3_enc
    ]
    X_raw = np.array(valores, dtype=np.float32).reshape(1, -1)
    X_sc  = sc.transform(X_raw)
    prob  = float(modelo.predict_proba(X_sc)[0][1])
    pred  = int(prob > 0.5)

    # Etiqueta de riesgo
    if prob >= 0.60:
        riesgo = '🔴 ALTO   — Intervención preventiva recomendada'
    elif prob >= 0.40:
        riesgo = '🟠 MODERADO — Seguimiento reforzado recomendado'
    else:
        riesgo = '🟢 BAJO   — Seguimiento estándar'

    # Valores SHAP locales
    sv_loc_raw = explainer.shap_values(X_sc)
    if isinstance(sv_loc_raw, list):
        sv_loc = sv_loc_raw[1][0]
    elif sv_loc_raw.ndim == 3:
        sv_loc = sv_loc_raw[0, :, 1]
    else:
        sv_loc = sv_loc_raw[0]

    # Mostrar resultado
    separador = '─' * 55
    print(separador)
    print('  RESULTADO DE LA PREDICCIÓN')
    print(separador)
    print(f'  ¿Readmisión predicha?  {"SÍ" if pred else "NO"}')
    print(f'  Probabilidad:          {prob*100:.1f} %')
    print(f'  Nivel de riesgo:       {riesgo}')
    print(separador)
    print('  TOP 8 FACTORES MÁS INFLUYENTES (SHAP)')
    print(separador)
    pares = sorted(zip(FEAT_LABELS, sv_loc),
                   key=lambda x: abs(x[1]), reverse=True)[:8]
    for lbl, val in pares:
        barra = '█' * int(abs(val) * 300)
        signo = '+' if val >= 0 else '-'
        dire  = '▲ riesgo' if val >= 0 else '▼ riesgo'
        print(f'  {lbl:<22} {signo}{abs(val):.4f}  {barra[:20]:<20} {dire}')
    print(separador)

    return {
        'readmitido':  pred,
        'probabilidad': round(prob, 4),
        'riesgo':      riesgo,
        'shap_values': dict(zip(FEAT_LABELS, sv_loc.round(4).tolist()))
    }


# ── Ejemplo 1: Paciente de alto riesgo ────────────────────────
print('\n📋 EJEMPLO 1 — Paciente con hospitalizaciones previas frecuentes')
r1 = predecir_paciente(
    time_in_hospital=8,
    n_inpatient=5,          # Múltiples hospitalizaciones previas
    n_lab_procedures=70,
    n_medications=25,
    A1Ctest_enc=2,          # HbA1c elevada
    change_enc=1,
    n_emergency=2
)

In [ ]:
# ── Ejemplo 2: Paciente de bajo riesgo ────────────────────────
print('\n📋 EJEMPLO 2 — Paciente con primer ingreso y buen control metabólico')
r2 = predecir_paciente(
    time_in_hospital=3,
    n_inpatient=0,          # Sin hospitalizaciones previas
    n_lab_procedures=30,
    n_medications=10,
    A1Ctest_enc=1,          # HbA1c normal
    change_enc=0,
    n_emergency=0
)

# ── Tu turno: personaliza los valores del paciente ────────────
print('\n📋 EJEMPLO 3 — Ingresa tus propios datos')
print('(Modifica los parámetros de la función y vuelve a ejecutar)')
r3 = predecir_paciente(
    time_in_hospital=1,
    n_lab_procedures=2,
    n_procedures=1,
    n_medications=1,
    n_outpatient=0,
    n_inpatient=1,
    n_emergency=0,
    age_enc=3,
    A1Ctest_enc=0,
    change_enc=1,
    diabetes_med_enc=1
)

## 💾 PASO 12 — Guardar modelos entrenados
Los archivos generados pueden descargarse desde el panel lateral de Colab.

In [ ]:
import joblib, os

os.makedirs('modelos', exist_ok=True)

# Guardar Random Forest y StandardScaler
joblib.dump(rf, 'modelos/rf_model.pkl')
joblib.dump(sc, 'modelos/scaler.pkl')
print('✓ Modelos guardados en la carpeta modelos/')

# Guardar figuras (ya guardadas en pasos anteriores)
print('✓ Figuras disponibles: eda_completo.png, comparacion_modelos.png, shap_explicabilidad.png')

# En Google Colab: descargar archivos
if IN_COLAB:
    from google.colab import files
    print('\n📥 Descargando modelos...')
    files.download('modelos/rf_model.pkl')
    files.download('modelos/scaler.pkl')
    print('\n📥 Descargando figuras...')
    files.download('eda_completo.png')
    files.download('comparacion_modelos.png')
    files.download('shap_explicabilidad.png')
    print('✓ Descarga completada.')
else:
    print('(En JupyterLab local: los archivos ya están en la carpeta del cuaderno.)')

print('\n✅ Pipeline completo ejecutado exitosamente.')

---
## 📚 Referencias

- Kaul, K. et al. (2022). *Predicting 30-day readmission in diabetic patients using ML.* Journal of Diabetes Science and Technology.
- Lundberg, S. M. y Lee, S. (2017). *A unified approach to interpreting model predictions.* NeurIPS.
- Strack, B. et al. (2014). *Impact of HbA1c on hospital readmission rates.* BioMed Research International.
- Murphy, K. P. (2022). *Probabilistic Machine Learning: An Introduction.* MIT Press.
- Zhang, A. et al. (2023). *Dive into Deep Learning.* Cambridge University Press.

---
**ACIF104 — Aprendizaje de Máquina NRC 2182 | UNAB 2026**